# YOLO-TRP Assignment (Google Colab)

Run cells in order. Before running, switch to **Runtime -> Change runtime type -> GPU**.

In [2]:
!nvidia-smi
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
print('Device count:', torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime not enabled. Switch runtime to GPU and reconnect.')

Wed Apr 22 04:23:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Clone Your Repository

Set `REPO_URL` and (optionally) `BRANCH` below.

In [1]:
import os, subprocess

REPO_URL = "https://github.com/ojhaprathmesh/Learnings.git"
BRANCH = "main"

# Always move to a stable directory first (important)
os.chdir("/content")

# Clean previous clone safely
if os.path.exists("/content/repo"):
    subprocess.run(["rm", "-rf", "/content/repo"], check=True)

# Clone
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/repo"],
    check=True
)

# Auto-detect assignment directory
candidates = [
    "/content/repo/_University/Semester-VI/DL_2026/Assignment-1",
    "/content/repo/_University/Semester-VI/DL_2026/Assignment_1",
    "/content/repo/Semester-VI/DL_2026/Assignment-1",
]
ASSIGNMENT_DIR = next((p for p in candidates if os.path.isdir(p)), None)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError("Could not locate Assignment-1 folder in cloned repo.")

os.chdir(ASSIGNMENT_DIR)
print("Working directory:", os.getcwd())

Working directory: /content/repo/_University/Semester-VI/DL_2026/Assignment-1


## Install Dependencies

This mirrors your local pipeline: install project deps, then force CUDA PyTorch wheels.

In [10]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA still unavailable after install.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ultralytics]
Looking in indexes: https://download.pytorch.org/whl/cu121
Torch: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8


## Drive dataset + persistent checkpoints

This notebook assumes your dataset lives on Google Drive (so you never re-upload).

It will:
- mount Drive
- patch `dataset.yaml` to point at your Drive dataset
- set env vars so **checkpoints + results** are written to Drive

Expected dataset layout at `DATA_ROOT`:
- `train/images`, `train/labels`
- `val/images`, `val/labels`
- `test/images`, `test/labels`

In [5]:
from google.colab import drive
import os
import yaml

# Mount Drive (no need to force remount unless it bugs out)
drive.mount('/content/drive')

# 1) Point this to your dataset folder on Drive
DATA_ROOT = '/content/drive/MyDrive/Colab Notebooks/DL_Dataset'

# 2) Where to persist outputs/checkpoints on Drive
PERSIST_ROOT = '/content/drive/MyDrive/Colab Notebooks/DL_Outputs'
RUNS_DIR = os.path.join(PERSIST_ROOT, 'runs', 'detect')
RESULTS_DIR = os.path.join(PERSIST_ROOT, 'results')
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Export env vars consumed by project scripts
os.environ['YOLO_RUNS_DIR'] = RUNS_DIR
os.environ['YOLO_RESULTS_DIR'] = RESULTS_DIR

print('YOLO_RUNS_DIR   =', os.environ['YOLO_RUNS_DIR'])
print('YOLO_RESULTS_DIR=', os.environ['YOLO_RESULTS_DIR'])

# Validate expected dataset structure
required = [
    'train/images', 'train/labels',
    'val/images', 'val/labels',
    'test/images', 'test/labels',
]
for p in required:
    full = os.path.join(DATA_ROOT, p)
    if not os.path.isdir(full):
        raise FileNotFoundError(f'Missing folder: {full}')

# Patch dataset.yaml (used by training + exploration + evaluation)
ASSIGNMENT_DIR = '/content/repo/_University/Semester-VI/DL_2026/Assignment-1'
YAML_PATH = os.path.join(ASSIGNMENT_DIR, 'dataset.yaml')
with open(YAML_PATH, 'r') as f:
    d = yaml.safe_load(f)

d['path'] = DATA_ROOT
d['train'] = 'train/images'
d['val'] = 'val/images'
d['test'] = 'test/images'

with open(YAML_PATH, 'w') as f:
    yaml.safe_dump(d, f, sort_keys=False)

print('\nUpdated dataset.yaml:\n')
print(open(YAML_PATH, 'r').read())

In [7]:
## Sanity check: confirm dataset.yaml + env vars are set

Mounted at /content/drive
Updated dataset.yaml:
path: /content/drive/MyDrive/Colab Notebooks/DL_Dataset
train: train/images
val: val/images
test: test/images
nc: 3
names:
  0: abiotic
  1: insect
  2: disease



In [8]:
import os
from pathlib import Path

print('YOLO_RUNS_DIR   =', os.environ.get('YOLO_RUNS_DIR'))
print('YOLO_RESULTS_DIR=', os.environ.get('YOLO_RESULTS_DIR'))

print('\nDataset.yaml:')
print(Path('dataset.yaml').read_text())

print('\nDrive checkpoint folders (should exist):')
if os.environ.get('YOLO_RUNS_DIR'):
    for p in ['baseline/weights', 'yolo_trp/weights']:
        print(' -', os.path.join(os.environ['YOLO_RUNS_DIR'], p))

OK    /content/drive/MyDrive/Colab Notebooks/DL_Dataset/train/images
OK    /content/drive/MyDrive/Colab Notebooks/DL_Dataset/train/labels
OK    /content/drive/MyDrive/Colab Notebooks/DL_Dataset/val/images
OK    /content/drive/MyDrive/Colab Notebooks/DL_Dataset/val/labels
OK    /content/drive/MyDrive/Colab Notebooks/DL_Dataset/test/images
OK    /content/drive/MyDrive/Colab Notebooks/DL_Dataset/test/labels


## Run Full Pipeline

In [11]:
%cd /content/repo/_University/Semester-VI/DL_2026/Assignment-1

# IMPORTANT: Cell 7 must run first so env vars + dataset.yaml are patched.

!python 00_sanitize_labels.py
!python 01_explore_dataset.py

# Train (checkpoints persist to Drive via YOLO_RUNS_DIR)
!python 02_train_baseline.py
!python 03_train_custom.py

# Evaluate + compare (outputs persist to Drive via YOLO_RESULTS_DIR)
!python 04_evaluate.py
!python 05_compare_results.py

/content
  YOLO Label Sanitizer
  Files scanned : 0
  Boxes clamped : 0
  Rows dropped  : 0

[1/4] Gathering annotation statistics …
Traceback (most recent call last):
  File "/content/repo/_University/Semester-VI/DL_2026/Assignment-1/01_explore_dataset.py", line 273, in <module>
    stats = gather_stats()
            ^^^^^^^^^^^^^^
  File "/content/repo/_University/Semester-VI/DL_2026/Assignment-1/01_explore_dataset.py", line 75, in gather_stats
    pairs = load_split(split_dir)
            ^^^^^^^^^^^^^^^^^^^^^
  File "/content/repo/_University/Semester-VI/DL_2026/Assignment-1/01_explore_dataset.py", line 51, in load_split
    for img_path in sorted(img_dir.iterdir()):
                    ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 1056, in iterdir
    for name in os.listdir(self):
                ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/repo/_University/Semester-VI/DL_2026/Assignment-1/train/train/images'
Creating

## Package Outputs

In [ ]:
# Package from persistent Drive outputs + key project files.
# (Note: Drive folders can be large; you can remove runs/ if size is too big.)
!zip -r assignment_outputs.zip "$YOLO_RESULTS_DIR" "$YOLO_RUNS_DIR" report *.py custom_model dataset.yaml requirements.txt
print('Created assignment_outputs.zip')

## Download Outputs

In [ ]:
from google.colab import files
files.download('assignment_outputs.zip')